In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
def load_dataset(filename):
    """
    Load a dataset from a CSV file.

    Parameters:
    filename (str): The path to the CSV file.

    Returns:
    tuple: A tuple containing the data matrix (numpy.ndarray) with shape (4, 150)
           and the corresponding labels (numpy.ndarray) with shape (150,).
    """
    tmp = []  # Initialize an empty list to store the data
    with open(filename, 'r') as f:  # Open the file in read mode
        for line in f:  # Iterate over each line in the file
            tmp.append(line.strip().split(","))  # Split the line by commas and append to the list
    tmp = np.array(tmp)  # Convert the list to a numpy array
    data = tmp[:, :4].astype(float).T  # Extract the first four columns, convert to float, and transpose
    labels = tmp[:, 4]  # Extract the fifth column as labels
    return data, labels  # Return the data and labels

X, y = load_dataset("iris.csv")


In [3]:
def plot_data(X, y):
    """
    Plot the distribution of each feature per label.

    Parameters:
    X (numpy.ndarray): The data matrix with shape (4, n_samples).
    y (numpy.ndarray): The labels array with shape (n_samples,).

    Returns:
    None
    """
    unique_labels = np.unique(y)  # Find unique labels in the labels array
    for i in range(X.shape[0]):  # Iterate over each feature
        plt.figure()  # Create a new figure for each feature
        for l in unique_labels:  # Iterate over each unique label
            mask = y == l  # Create a boolean mask for the current label
            plt.hist(X[i, mask], bins=10, alpha=0.6, density=True, label=l)  # Plot histogram for the current feature and label
        plt.legend()
        plt.show()  # Display the plot

    for i in range(X.shape[0]):  # Iterate over each feature
        for j in range(i+1, X.shape[0]):
            plt.figure()
            for l in unique_labels:
                mask = y == l
                plt.scatter(X[i, mask], X[j, mask], alpha=0.6, label=l)
            plt.legend()
            plt.show()

# Plot the distribution of each feature per label and scatter plots of feature pairs
# plot_data(X, y)


In [4]:
def center_data(X):
    """
    Center the dataset by subtracting the mean of each feature.

    Parameters:
    X (numpy.ndarray): The data matrix with shape (4, n_samples).

    Returns:
    numpy.ndarray: The centered data matrix with the same shape as X.
    """
    mu = np.mean(X, axis=1).reshape(-1, 1)  # Calculate the mean of each feature and reshape to (4, 1)
    return X - mu  # Center the dataset by subtracting the mean from each feature


# Center the dataset
Xc = center_data(X)

In [5]:
def covariance_matrix(Xc):
    """
    Compute the covariance matrix of the dataset.

    Parameters:
    X (numpy.ndarray): The data matrix with shape (4, n_samples).

    Returns:
    numpy.ndarray: The covariance matrix with shape (4, 4).
    """
    N = Xc.shape[1]  # Get the number of samples
    return (Xc @ Xc.T) / float(N)  # Compute the covariance matrix by matrix multiplication and normalization

# Compute the covariance matrix
C = covariance_matrix(Xc)
print(C)

[[ 0.68112222 -0.04215111  1.26582     0.51282889]
 [-0.04215111  0.18871289 -0.32745867 -0.12082844]
 [ 1.26582    -0.32745867  3.09550267  1.286972  ]
 [ 0.51282889 -0.12082844  1.286972    0.57713289]]


In [6]:
def features_var_and_std(Xc):
    """
    Compute the variance and standard deviation of each feature in the dataset.

    Parameters:
    X (numpy.ndarray): The data matrix with shape (4, n_samples).

    Returns:
    tuple: A tuple containing the variance (numpy.ndarray) and standard deviation (numpy.ndarray) of each feature.
    """
    return np.var(Xc, axis=1), np.std(Xc, axis=1)  # Compute the variance of each feature along the samples axis

# Compute the variance and standard deviation of each feature
var, std = features_var_and_std(Xc)
print(var)  # Print the variance of each feature
print(std)  # Print the standard deviation of each feature

# Extract the diagonal elements of the covariance matrix (variances)
var = np.diag(C)
std = np.sqrt(var)
print(var)  # Print the variances
print(std)  # Print the standard deviations

[0.68112222 0.18871289 3.09550267 0.57713289]
[0.82530129 0.43441097 1.75940407 0.75969263]
[0.68112222 0.18871289 3.09550267 0.57713289]
[0.82530129 0.43441097 1.75940407 0.75969263]


In [7]:
def covariance_per_class(Xc, y):
    """
    Compute the covariance matrix for each class in the dataset.

    Parameters:
    Xc (numpy.ndarray): The centered data matrix with shape (4, n_samples).
    y (numpy.ndarray): The labels array with shape (n_samples,).

    Returns:
    dict: A dictionary where keys are class labels and values are the covariance matrices (numpy.ndarray) for each class.
    """
    unique_labels = np.unique(y)  # Find unique labels in the labels array
    cov_matrices = {}  # Initialize an empty dictionary to store covariance matrices
    for label in unique_labels:  # Iterate over each unique label
        Xc_class = Xc[:, y == label]  # Extract the data for the current class
        cov_matrices[label] = covariance_matrix(Xc_class)  # Compute the covariance matrix for the current class
    return cov_matrices  # Return the dictionary of covariance matrices

# Example usage
cov_matrices = covariance_per_class(Xc, y)
for label, cov_matrix in cov_matrices.items():
    print(f"Covariance matrix for class {label}:\n{cov_matrix}\n")

Covariance matrix for class Iris-setosa:
[[ 0.82289111 -0.21313956  1.93854533  0.80838178]
 [-0.21313956  0.27820978 -0.83958667 -0.34425689]
 [ 1.93854533 -0.83958667  5.301172    2.19480133]
 [ 0.80838178 -0.34425689  2.19480133  0.91972844]]

Covariance matrix for class Iris-versicolor:
[[ 0.26969111  0.05685378  0.22575867  0.06640178]
 [ 0.05685378  0.17906044 -0.06324133  0.00398444]
 [ 0.22575867 -0.06324133  0.468404    0.13522667]
 [ 0.06640178  0.00398444  0.13522667  0.05436844]]

Covariance matrix for class Iris-virginica:
[[ 0.95078444  0.02983244  1.633156    0.66370311]
 [ 0.02983244  0.10886844 -0.079548   -0.02221289]
 [ 1.633156   -0.079548    3.516932    1.530888  ]
 [ 0.66370311 -0.02221289  1.530888    0.75730178]]



###
Features show significant differences across classes. Notably:
    Feature 3 (third diagonal entry):
        Iris-setosa has a much higher variance (5.30) compared to Iris-versicolor (0.47) and Iris-virginica (3.52).
        Covariances with other features (e.g., Feature 1 and 4) are also stronger in setosa and virginica than in versicolor.
    Feature 4 (fourth diagonal entry):
        Iris-versicolor exhibits a much lower variance (0.05) compared to setosa (0.92) and virginica (0.76).
    Feature 2:
        Iris-virginica has a lower variance (0.11) than setosa (0.28) and versicolor (0.18).
These differences suggest that petal-related features (Features 3 and 4) are the most discriminative, aligning with known Iris species distinctions.
###

In [8]:
def var_and_std_per_class(Xc, y):
    """
    Compute the variance and standard deviation for each class in the dataset.

    Parameters:
    Xc (numpy.ndarray): The centered data matrix with shape (4, n_samples).
    y (numpy.ndarray): The labels array with shape (n_samples,).

    Returns:
    tuple: A tuple containing two dictionaries:
           - variances: A dictionary where keys are class labels and values are the variances (numpy.ndarray) for each class.
           - stds: A dictionary where keys are class labels and values are the standard deviations (numpy.ndarray) for each class.
    """
    unique_labels = np.unique(y)  # Find unique labels in the labels array
    variances = {}  # Initialize an empty dictionary to store variances
    stds = {}  # Initialize an empty dictionary to store standard deviations
    for label in unique_labels:  # Iterate over each unique label
        Xc_class = Xc[:, y == label]  # Extract the data for the current class
        variances[label], stds[label] = features_var_and_std(Xc_class)  # Compute the variance and standard deviation for the current class
    return variances, stds  # Return the dictionaries of variances and standard deviations

# Compute the variance and standard deviation for each class
variances, stds = var_and_std_per_class(Xc, y)
for label, var in variances.items():
    print(f"Variances for class {label}:\n{var}\n")  # Print the variances for each class

for label, std in stds.items():
    print(f"Standard deviations for class {label}:\n{std}\n")  # Print the standard deviations for each class

Variances for class Iris-setosa:
[0.121764 0.140816 0.029556 0.010884]

Variances for class Iris-versicolor:
[0.261104 0.0965   0.2164   0.038324]

Variances for class Iris-virginica:
[0.396256 0.101924 0.298496 0.073924]

Standard deviations for class Iris-setosa:
[0.34894699 0.37525458 0.17191859 0.10432641]

Standard deviations for class Iris-versicolor:
[0.51098337 0.31064449 0.46518813 0.19576517]

Standard deviations for class Iris-virginica:
[0.62948868 0.31925538 0.54634787 0.27188968]



Iris-virginica generally shows the highest variance and standard deviation across most features, indicating more variability within this class.
Iris-setosa shows the lowest variance and standard deviation in petal-related features (petal length and petal width), indicating less variability within this class.
Iris-versicolor has intermediate values for most features but shows the lowest variance and standard deviation in sepal width.
These differences suggest that petal-related features (Features 3 and 4) are the most discriminative, aligning with known distinctions between Iris species.

In [9]:
def class_means(Xc, y):
    """
    Compute the mean vector for each class in the dataset.

    Parameters:
    Xc (numpy.ndarray): The centered data matrix with shape (4, n_samples).
    y (numpy.ndarray): The labels array with shape (n_samples,).

    Returns:
    dict: A dictionary where keys are class labels and values are the mean vectors (numpy.ndarray) for each class.
    """
    unique_labels = np.unique(y)  # Find unique labels in the labels array
    means = {}  # Initialize an empty dictionary to store mean vectors
    for label in unique_labels:  # Iterate over each unique label
        Xc_class = Xc[:, y == label]  # Extract the data for the current class
        means[label] = np.mean(Xc_class, axis=1)  # Compute the mean vector for the current class
    return means  # Return the dictionary of mean vectors

# Compute the mean vector for each class
means = class_means(Xc, y)
for label, mean in means.items():
    print(f"Mean vector for class {label}:\n{mean}\n")  # Print the mean vector for each class

Mean vector for class Iris-setosa:
[-0.83733333  0.37066667 -2.296      -0.95333333]

Mean vector for class Iris-versicolor:
[ 0.09266667 -0.28733333  0.502       0.12666667]

Mean vector for class Iris-virginica:
[ 0.74466667 -0.08333333  1.794       0.82666667]



Iris-setosa generally shows lower mean values for petal-related features (petal length and petal width) compared to the other classes.
Iris-virginica shows higher mean values for petal-related features, indicating larger petals.
Iris-versicolor has intermediate mean values for most features but shows the lowest mean sepal width.
These differences suggest that petal-related features (Features 3 and 4) are the most discriminative, aligning with known distinctions between Iris species.